5.1 In this exercise, you will compare two investment strategies involving the same stock:

* holding 100 shares, and
* holding 100 shares combined with one European put option (a protective put, involving 100 shares).

The stock currently trades at 80, with an annual volatility of 35%, a risk-free rate of 6%, and an expected annual growth rate of 15%. The put option has an exercise price of $75 and a maturity of 0.5 years. 

* Estimate the value of the put option using the Black–Scholes model. Note that this model assumes growth of the stock at the risk-free rate.
* Simulate and compare the terminal values of each portfolio at maturity, by using the annual growth rate for the stock price simulation.
* Analyze how the inclusion of the put option affects the portfolio’s performance and risk exposure. 


In [ ]:
# Import Neccessary Libraries
import numpy as np
import pandas as pd
import scipy.stats as sp
import matplotlib.pyplot as plt

In [ ]:
def summarystats(Outp):
    '''Generic code to compute descriptive statistics for each column in a data frame Outp'''
    Mean = Outp.mean()
    StDev = Outp.std(ddof=1)
    Percent05 = Outp.quantile(0.05)
    Percent50 = Outp.quantile(0.50)
    Percent95 = Outp.quantile(0.95)
    StdErr = Outp.sem()
    Min = Outp.min()
    Max = Outp.max()

    Prob = 0.95
    Tails = (1 - Prob) / 2
    z = sp.norm.ppf(1 - Tails)
    CI_low = Mean - z * StdErr
    CI_high = Mean + z * StdErr

    Sample_stats = pd.concat(
        [Mean, StDev, Percent05, Percent50, Percent95, StdErr, CI_low, CI_high, Min, Max],
        axis=1
    )
    Sample_stats.columns = [
        'Sample Mean', 'Sample St Dev', 'Percentile 5', 'Median', 'Percentile 95',
        'Std Error', 'CI low', 'CI high', 'Min', 'Max'
    ]
    return Sample_stats.round(3)


def descriptivehistogram(Outp):
    '''
    Generic code to plot a histogram for each column in the output dataframe Outp
    '''
    for column in list(Outp.columns):
        plt.clf()
        plt.hist(Outp[column], bins='auto', edgecolor='black', density=True)
        plt.xlabel(column)
        plt.ylabel('Frequency')
        plt.title(f'Histogram of {column}')
        plt.show()

In [ ]:
def black_scholes_put(S0, K, r, sigma, T):
    '''
    Black-Scholes value of a European put
    '''
    d1 = (np.log(S0 / K) + (r + 0.5 * sigma**2) * T) / (sigma * np.sqrt(T))
    d2 = d1 - sigma * np.sqrt(T)

    put_value = K * np.exp(-r * T) * sp.norm.cdf(-d2) - S0 * sp.norm.cdf(-d1)
    return put_value, d1, d2


In [ ]:
def simulationfunction(controls, inputs):
    '''
    Generic function to structure the simulation process
    '''
    # A. Standardized controls for the simulation
    seed = controls[0]   # Will a common seed be used?
    n = controls[1]      # Sample size

    # B. Model-dependent inputs
    S0 = inputs[0]       # Current stock price
    sigma = inputs[1]    # Annual volatility
    r = inputs[2]        # Risk-free rate
    mu = inputs[3]       # Expected annual growth rate
    K = inputs[4]        # Put strike price
    T = inputs[5]        # Time to maturity
    shares = inputs[6]   # Number of shares

    # C. Use a seed or not
    if seed:
        np.random.seed(100)

    # D. Black-Scholes put value
    put_price, d1, d2 = black_scholes_put(S0, K, r, sigma, T)

    # E. Generate random stock prices at maturity using expected growth rate mu
    Z = np.random.normal(0, 1, n)
    ST = S0 * np.exp((mu - 0.5 * sigma**2) * T + sigma * np.sqrt(T) * Z)

    # F. Portfolio terminal values
    stock_only_terminal = shares * ST
    protective_put_terminal = shares * ST + shares * np.maximum(K - ST, 0)

    # G. Initial portfolio costs
    stock_only_initial = shares * S0
    protective_put_initial = shares * S0 + shares * put_price

    # H. Profit at maturity
    stock_only_profit = stock_only_terminal - stock_only_initial
    protective_put_profit = protective_put_terminal - protective_put_initial

    # I. Store outputs
    Outp = pd.DataFrame({
        'Stock Price at Maturity': ST,
        'Stock Only Terminal Value': stock_only_terminal,
        'Protective Put Terminal Value': protective_put_terminal,
        'Stock Only Profit': stock_only_profit,
        'Protective Put Profit': protective_put_profit
    })

    return Outp, put_price, stock_only_initial, protective_put_initial

In [ ]:
# Run the simulation
controls = [True, 100000]

# Inputs:
# S0, sigma, r, mu, K, T, shares
inputs = [80, 0.35, 0.06, 0.15, 75, 0.5, 100]

Outp, put_price, stock_only_initial, protective_put_initial = simulationfunction(controls, inputs)

print(f"Black-Scholes put value per share: ${put_price:.4f}")
print(f"Cost of 1 put contract on 100 shares: ${100 * put_price:.2f}")
print(f"Initial cost of stock-only portfolio: ${stock_only_initial:.2f}")
print(f"Initial cost of protective-put portfolio: ${protective_put_initial:.2f}")

display(summarystats(Outp))

In [ ]:
# Additional risk metrics
risk_metrics = pd.DataFrame({
    'Metric': [
        'Probability stock-only loses money',
        'Probability protective put loses money',
        'Worst terminal value: stock only',
        'Worst terminal value: protective put',
        'Downside floor of protective put terminal value'
    ],
    'Value': [
        (Outp['Stock Only Profit'] < 0).mean(),
        (Outp['Protective Put Profit'] < 0).mean(),
        Outp['Stock Only Terminal Value'].min(),
        Outp['Protective Put Terminal Value'].min(),
        100 * inputs[4]
    ]
})

display(risk_metrics)


In [ ]:
# Histograms
descriptivehistogram(Outp[['Stock Only Terminal Value', 'Protective Put Terminal Value']])
descriptivehistogram(Outp[['Stock Only Profit', 'Protective Put Profit']])

# Side-by-side comparison summary
comparison = pd.DataFrame({
    'Portfolio': ['Stock Only', 'Protective Put'],
    'Initial Cost': [stock_only_initial, protective_put_initial],
    'Expected Terminal Value': [
        Outp['Stock Only Terminal Value'].mean(),
        Outp['Protective Put Terminal Value'].mean()
    ],
    'Expected Profit': [
        Outp['Stock Only Profit'].mean(),
        Outp['Protective Put Profit'].mean()
    ],
    'Std Dev of Profit': [
        Outp['Stock Only Profit'].std(ddof=1),
        Outp['Protective Put Profit'].std(ddof=1)
    ],
    '5th Percentile Profit': [
        Outp['Stock Only Profit'].quantile(0.05),
        Outp['Protective Put Profit'].quantile(0.05)
    ],
    'Minimum Profit': [
        Outp['Stock Only Profit'].min(),
        Outp['Protective Put Profit'].min()
    ]
})

display(comparison.round(3))

 ## Put value per share is 4.4539, so cost of protecting 100 shares is 445.39

## The put does not maximize average profit, but it substantially improves risk protection. If your goal is pure return, holding only the stock is better. If your goal is to stay invested while controlling losses, the protective put is the safer strategy.

5.2 A sports apparel company has secured the rights to sell official jerseys during the upcoming Soccer World Cup. The company must decide how many jerseys to order before the tournament begins. Each jersey costs 30 to produce and sells for 75 during the World Cup. However, any unsold jerseys lose most of their value once the tournament ends—they can be sold at a discount price of 37.50 during a short post-tournament clearance period, but any remaining inventory after that has no value. The key challenge is that demand is uncertain: the number of jerseys sold during the World Cup may vary significantly depending on how far popular teams advance and fan enthusiasm.

The goal is to determine the optimal order quantity that maximizes the company’s expected profit in the case where the 2 demand streams are independent and in the case where they are negatively correlated.

In [ ]:
def expected_profit(Q, D1, D2):
    '''
    Profit for order quantity Q
    D1 = tournament demand
    D2 = post-tournament clearance demand
    '''
    sold_tournament = np.minimum(Q, D1)
    leftover_after_tournament = np.maximum(Q - D1, 0)
    sold_clearance = np.minimum(leftover_after_tournament, D2)

    revenue = 75 * sold_tournament + 37.5 * sold_clearance
    cost = 30 * Q
    profit = revenue - cost

    return profit

In [ ]:
def simulationfunction(controls, inputs):
    '''
    Generic function to structure the simulation process
    '''
    # A. Standardized controls for the simulation
    seed = controls[0]
    n = controls[1]

    # B. Model-dependent inputs
    Q = inputs[0]                        # candidate order quantities
    D1_vals = np.array(inputs[1][0])     # tournament demand support
    D1_probs = np.array(inputs[1][1])    # tournament demand probabilities
    D2_vals = np.array(inputs[2][0])     # clearance demand support
    D2_probs = np.array(inputs[2][1])    # clearance demand probabilities

    # C. Use a seed or not
    if seed:
        np.random.seed(100)

    # -----------------------------
    # 1. Independent demand streams
    # -----------------------------
    D1_ind = np.random.choice(D1_vals, size=n, p=D1_probs)
    D2_ind = np.random.choice(D2_vals, size=n, p=D2_probs)

    # -----------------------------------
    # 2. Negatively correlated demand case
    # -----------------------------------
    # We create strong negative dependence by using one uniform draw u:
    # D1 from F^{-1}(u), D2 from G^{-1}(1-u)
    u = np.random.rand(n)

    # Tournament demand from u
    cdf1 = np.cumsum(D1_probs)
    D1_neg = np.select(
        [u <= cdf1[0], u <= cdf1[1], u <= cdf1[2], u <= cdf1[3], u <= cdf1[4], u <= cdf1[5]],
        D1_vals
    )

    # Clearance demand from 1-u, using ascending order of D2 values
    D2_sort_idx = np.argsort(D2_vals)
    D2_vals_sorted = D2_vals[D2_sort_idx]
    D2_probs_sorted = D2_probs[D2_sort_idx]
    cdf2 = np.cumsum(D2_probs_sorted)
    v = 1 - u

    D2_neg = np.select(
        [v <= cdf2[0], v <= cdf2[1], v <= cdf2[2], v <= cdf2[3], v <= cdf2[4], v <= cdf2[5]],
        D2_vals_sorted
    )

    # D. Compute simulated profits for every candidate Q
    Profit_ind = pd.DataFrame()
    Profit_neg = pd.DataFrame()

    for q in Q:
        Profit_ind[f'Q={q}'] = expected_profit(q, D1_ind, D2_ind)
        Profit_neg[f'Q={q}'] = expected_profit(q, D1_neg, D2_neg)

    return Profit_ind, Profit_neg, D1_neg, D2_neg

In [ ]:
# Inputs

controls = [True, 200000]

Q_candidates = np.arange(0, 20001, 500)

D1_values = [4500, 7500, 8500, 11000, 12500, 15000]
D1_probs  = [0.15, 0.10, 0.20, 0.35, 0.15, 0.05]

D2_values = [1000, 2000, 5000, 8000, 5500, 10000]
D2_probs  = [0.15, 0.25, 0.25, 0.10, 0.15, 0.10]

inputs = [
    Q_candidates,
    [D1_values, D1_probs],
    [D2_values, D2_probs]
]

In [ ]:
# Run simulation
Profit_ind, Profit_neg, D1_neg, D2_neg = simulationfunction(controls, inputs)

# Display summaries
print("Independent-demand case")
display(summarystats(Profit_ind).sort_values(by='Sample Mean', ascending=False).head(10))

print("Negatively correlated-demand case")
display(summarystats(Profit_neg).sort_values(by='Sample Mean', ascending=False).head(10))

# Best order quantities from simulation
best_ind = summarystats(Profit_ind)['Sample Mean'].idxmax()
best_neg = summarystats(Profit_neg)['Sample Mean'].idxmax()

print("Best simulated order quantity under independence:", best_ind)
print("Best simulated order quantity under negative correlation:", best_neg)

# Show realized negative correlation in the simulation
corr = np.corrcoef(D1_neg, D2_neg)[0, 1]
print("Approximate simulated correlation in negative-correlation case:", round(corr, 3))

# Histograms of the best policies
descriptivehistogram(Profit_ind[[best_ind]])
descriptivehistogram(Profit_neg[[best_neg]])

## When tournament and clearance demand are assumed independent, the profit-maximizing order quantity is 12,500 jerseys, with expected profit 415,218.75. When the two demand streams are modeled as negatively correlated, the optimal order quantity increases to 13,000 jerseys, with expected profit $452,812.50. The reason is that negative correlation makes high leftover inventory more likely to be absorbed by stronger post-tournament clearance demand, reducing downside risk from overordering.